# 03. 카테고리와 셀러 — 어디부터 손대야 하는가02에서 "지연되면 점수가 떨어진다"를 확인했다면, 이제 **어디서 가장 크게 떨어지는가** 를 봅니다.개선 우선순위를 정하는 것이 목적입니다.이 노트북의 후반부가 **SQL 실력이 가장 많이 드러나는 구간** 입니다. 시간을 제일 많이 쓰세요.

In [ ]:
import duckdb, pandas as pdimport matplotlib, matplotlib.pyplot as pltpd.set_option("display.max_columns", 50)# 한글 깨짐 방지for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR", "Noto Sans KR"]:    if _f in {f.name for f in matplotlib.font_manager.fontManager.ttflist}:        plt.rcParams["font.family"] = _f        breakplt.rcParams["axes.unicode_minus"] = False# DuckDB는 CSV를 그대로 읽습니다. 적재 과정이 따로 없습니다.con = duckdb.connect()con.execute(open("../sql/base.sql", encoding="utf-8").read().format(data="../data/raw"))def q(sql):    """SQL을 실행해 pandas DataFrame으로 돌려줍니다."""    return con.sql(sql).df()q("SELECT count(*) AS 주문수 FROM order_fact")

## 1. 카테고리별 지연 타격도같은 지연이라도 상품군에 따라 고객 반응이 다릅니다.`FILTER (WHERE ...)` 를 쓰면 한 번의 집계로 조건별 값을 나란히 뽑을 수 있습니다.

In [ ]:
cat = q("""SELECT category AS 카테고리,       count(*) AS 주문수,       round(avg(review_score) FILTER (WHERE NOT is_late), 3) AS 정시_평균점수,       round(avg(review_score) FILTER (WHERE is_late), 3)     AS 지연_평균점수,       round(avg(review_score) FILTER (WHERE NOT is_late)           - avg(review_score) FILTER (WHERE is_late), 3)     AS 하락폭FROM order_factWHERE has_review AND is_late IS NOT NULL AND category <> 'unknown'GROUP BY 카테고리HAVING count(*) >= 200 AND count(*) FILTER (WHERE is_late) >= 30ORDER BY 하락폭 DESC NULLS LASTLIMIT 15""")cat

In [ ]:
cat.sort_values("하락폭").plot(x="카테고리", y="하락폭", kind="barh",                               figsize=(9, 6), legend=False, color="#1F6F6B")plt.title("정시 대비 점수 하락폭 (클수록 지연에 민감)"); plt.tight_layout(); plt.show()

**주의** — `HAVING count(*) >= 200` 의 200은 제가 임의로 정한 값입니다.기준을 바꾸면 순위가 바뀝니다. 100, 500으로도 돌려 보고 순위가 뒤집히는지 확인한 뒤,어떤 기준을 왜 택했는지 `docs/METRICS.md` 에 적으세요.

## 2. 【직접 작성】 셀러 성과 순위 — 윈도우 함수**목표**: "관리가 필요한 셀러"를 자동으로 뽑아내는 쿼리.순위만이 아니라 **전체 대비 어디쯤인지** 도 같이 보여야 판단 근거가 됩니다.### 만들 컬럼| 컬럼 | 쓸 함수 ||---|---|| 주문수, 평균점수, 지연율, 평균지연일 | 일반 집계 || 지연율 사분위 (1~4) | `NTILE(4) OVER (ORDER BY 지연율)` || 평균점수 순위 | `RANK() OVER (ORDER BY 평균점수)` || 전체 평균점수 (비교용으로 각 행에 붙이기) | `avg(평균점수) OVER ()` |### 윈도우 함수가 처음이라면일반 집계(`GROUP BY`)는 여러 행을 **한 줄로 합칩니다.**윈도우 함수는 행을 그대로 두고 **옆에 계산 결과를 한 칸 더 붙입니다.**```sqlavg(x) OVER ()                 -- 전체 평균을 모든 행에 붙임RANK() OVER (ORDER BY x DESC)  -- x 기준 순위를 붙임NTILE(4) OVER (ORDER BY x)     -- x 기준으로 4등분해 1~4 번호를 붙임```### 구조 힌트집계를 먼저 하고, 그 결과에 윈도우 함수를 얹어야 하므로 CTE 두 단계로 나눕니다.```sqlWITH s AS (    SELECT seller_id,           count(*) AS 주문수,           round(avg(review_score), 3) AS 평균점수,           round(100.0 * count(*) FILTER (WHERE is_late)                 / count(*) FILTER (WHERE is_late IS NOT NULL), 1) AS 지연율    FROM order_fact    WHERE has_review AND is_late IS NOT NULL    GROUP BY seller_id    HAVING count(*) >= 30        -- ← 기준을 정하고 근거를 남길 것)SELECT *,       ...  -- 여기에 NTILE / RANK / avg(...) OVER () 추가FROM sORDER BY 지연율 DESC```### 함께 기록할 한계`order_fact` 의 `seller_id` 는 **주문 내 최고가 상품의 셀러** 입니다 (`sql/base.sql` 참고).여러 셀러가 섞인 주문의 책임을 한 명에게 몰아주고 있다는 뜻이니, 결론에 이 한계를 함께 적으세요.

In [ ]:
# 여기에 작성하세요# seller = q("""# WITH s AS (#     ...# )# SELECT ...# """)# seller.head(20)

## 3. 대시보드에 붙이기위 쿼리가 완성되면 `app.py` 의 "추가할 자리" 주석 아래에 아래처럼 한 번만 부르면 화면에 붙습니다.```pythonblock(    "셀러 성과 순위",    """<완성한 SQL 그대로>""",    definition="- **지연율 사분위**: 전체 셀러를 지연율 기준 4등분 ...",    caveat="주문 30건 미만 셀러는 제외했습니다. 대표 셀러는 주문 내 최고가 상품 기준입니다.",)```

## 결론 작성> 셀러 **[   ]** 곳 중 지연율 하위 사분위 **[   ]** 곳이 전체 지연 주문의 **[  ]** %를 차지한다.> 이 **[   ]** 곳만 관리해도 지연 주문의 **[  ]** %가 개선 대상에 들어온다.카테고리는:> **[   ]** 카테고리가 지연에 가장 민감해 정시 대비 **[   ]** 점 하락하고, **[   ]** 는 **[   ]** 점으로 가장 둔감하다.